# Cartórios do Brasil · Brazilian Notary and Registry Offices

A sample of about 1,000 of the 16,266 offices in the CNJ registry (Jul/2026), stratified by state, with their services,
the acts and revenue they reported per semester since 2005 and the decisions of the CNJ Corregedoria.

Data: [Kaggle](https://www.kaggle.com/datasets/caiquedourado/brazilian-notary-registry-offices) ·
code and data dictionary: [GitHub](https://github.com/CaiqueDourado/brazilian-notary-registry-offices)

In [1]:
import glob, json, os
import duckdb

# Kaggle: /kaggle/input/…; GitHub clone: data/
found = glob.glob('/kaggle/input/**/serventias.parquet', recursive=True) or glob.glob('../data/serventias.parquet')
base = os.path.dirname(found[0])
SERV = f"read_parquet('{base}/serventias.parquet')"
ATR = f"read_csv_auto('{base}/atribuicoes.csv')"
ARR = f"read_csv_auto('{base}/arrecadacao.csv')"
DEC = f"read_csv_auto('{base}/decisoes.csv')"
stats = json.load(open(f'{base}/stats.json', encoding='utf-8'))
q = lambda sql: duckdb.sql(sql).df()
q(f'SELECT count(*) serventias, count(DISTINCT uf) ufs FROM {SERV}')

,serventias,ufs
0,1009,27


## All offices, from `stats.json`

The sample has ~1,000 offices; `stats.json` keeps the counts of all 16,266.

In [2]:
{k: stats[k] for k in ('serventias_2026', 'serventias_por_status', 'serventias_por_tipo', 'atribuicoes')}

{'serventias_2026': 16266,
 'serventias_por_status': {'Ativo': 12203, 'Inativo': 3881, 'Extinto': 182},
 'serventias_por_tipo': {'PRIVATIZADO': 12789,
  'OFICIALIZADO': 3431,
  'PUBLICO': 46},
 'atribuicoes': {'Notas': 9699,
  'Registro Civil das Pessoas Naturais': 8879,
  'Protesto de Títulos': 4465,
  'Registro de Títulos e Documentos': 4413,
  'Registro Civil das Pessoas Jurídicas': 4406,
  'Registro de Imóveis': 4136,
  'Registro de Contratos Marítimos': 645,
  'Registro de Distribuição': 469,
  'Registro de Interdições e Tutelas': 109}}

## Offices per state and status (sample)

In [3]:
q(f'''SELECT uf, count(*) serventias, count(*) FILTER (status = 'Ativo') ativas,
      count(*) FILTER (situacao = 'VAGO') vagas FROM {SERV} GROUP BY ALL ORDER BY serventias DESC LIMIT 10''')

,uf,serventias,ativas,vagas
0,MG,198,151,82
1,BA,111,47,47
2,SP,103,95,10
3,PR,80,60,23
4,RS,51,49,15
5,GO,46,31,26
6,CE,45,35,6
7,SC,42,32,9
8,PE,38,31,11
9,RJ,38,27,5


## Services

One office can provide several services: in small towns one *Ofício Único* is notary, civil registry and real estate
registry at once.

In [4]:
q(f'''SELECT n_atribuicoes, count(*) serventias FROM
      (SELECT cns, count(*) n_atribuicoes FROM {ATR} GROUP BY 1) GROUP BY 1 ORDER BY 1''')

,n_atribuicoes,serventias
0,1,261
1,2,394
2,3,79
3,4,48
4,5,47
5,6,67
6,7,20
7,8,7


## Revenue per year

Nominal R$, not adjusted for inflation. Early years may be one period per year; later ones are semesters. Only rows
with `situacao = 'informado'` have a value.

In [5]:
q(f'''SELECT year(inicio) ano, count(DISTINCT cns) serventias, round(sum(valor) / 1e6, 1) r_milhoes,
      round(median(valor)) mediana_periodo FROM {ARR} WHERE situacao = 'informado' GROUP BY 1 ORDER BY 1''')

,ano,serventias,r_milhoes,mediana_periodo
0,2005,909,232.5,25736.0
1,2006,909,255.8,31914.0
2,2007,909,311.2,17063.0
3,2008,894,375.0,24284.0
4,2009,886,412.7,26886.0
5,2010,880,478.1,32789.0
6,2011,877,569.8,41583.0
7,2012,874,643.8,51612.0
8,2013,872,748.5,66292.0
9,2014,871,898.9,84038.0


Over all offices (`stats.json`), in R$ billions:

In [6]:
{y: round(v / 1e9, 2) for y, v in stats['valor_por_ano'].items()}

{'2005': 3.71,
 '2006': 4.2,
 '2007': 4.76,
 '2008': 5.81,
 '2009': 7.34,
 '2010': 7.74,
 '2011': 8.79,
 '2012': 10.12,
 '2013': 12.18,
 '2014': 14.1,
 '2015': 14.79,
 '2016': 15.42,
 '2017': 16.19,
 '2018': 17.42,
 '2019': 18.25,
 '2020': 17.96,
 '2021': 24.64,
 '2022': 26.61,
 '2023': 28.53,
 '2024': 32.26,
 '2025': 38.0,
 '2026': 17.48}

## Revenue per act, by service mix (last full year)

In [7]:
q(f'''WITH a AS (SELECT cns, sum(valor) valor, sum(atos) atos FROM {ARR}
                 WHERE situacao = 'informado' AND year(inicio) = 2025 GROUP BY 1),
         r AS (SELECT cns, bool_or(atribuicao = 'Registro de Imóveis') imoveis FROM {ATR} GROUP BY 1)
      SELECT imoveis, count(*) serventias, round(median(valor)) mediana_r, round(sum(valor) / sum(atos), 2) r_por_ato
      FROM a JOIN r USING (cns) WHERE atos > 0 GROUP BY 1''')

,imoveis,serventias,mediana_r,r_por_ato
0,False,500,656543.0,49.83
1,True,237,1563046.0,152.12


## Corregedoria decisions

`texto` is kept only for standard texts; `texto_omitido` marks texts left out because they may name people.

In [8]:
q(f'''SELECT status, count(*) decisoes, count(texto) com_texto, count(*) FILTER (texto_omitido) omitidos
      FROM {DEC} GROUP BY 1 ORDER BY decisoes DESC''')

,status,decisoes,com_texto,omitidos
0,VAGO,2077,1658,419
1,PROVIDO,1397,1228,169
2,CONVERSÃO EM DILIGÊNCIA,678,591,87
3,INATIVA,134,120,14
4,EXCLUÍDO DA RELAÇÃO ORIGINÁRIA,32,20,12
5,SEM DECISÃO ORIGINAL,31,31,0
6,PENDÊNCIA JUDICIAL CAPAZ DE AFASTAR A ANÁLISE ...,24,1,23
7,SOB INTERVENÇÃO,21,19,2
8,EXTINTO,7,2,5
9,VAGO - SUB JUDICE,1,0,1


In [9]:
q(f'SELECT texto, count(*) n FROM {DEC} WHERE texto IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 8')

,texto,n
0,Status alterado pela corregedoria estadual.,1398
1,Provido por meio de Concurso de provas e títul...,529
2,"Essa Serventia foi declarada vaga, pois seu ti...",334
3,Designado/Nomeado Titular/Delegado até 05/10/1...,246
4,Serventia inativada,85
5,Aguardando a alteração de status pelo CNJ após...,76
6,Em razão de declaração válida do Tribunal de J...,75
7,Em razão de declaração válida do Tribunal de J...,68


## One office as nested JSON

In [10]:
rec = json.loads(open(f'{base}/serventias.jsonl', encoding='utf-8').readline())
{k: (v[:3] if isinstance(v, list) else v) for k, v in rec.items()}

{'cns': '000869',
 'denominacao': 'Primeiro Tabelionato de Notas da Comarca de Acrelândia',
 'status': 'Inativo',
 'tipo': 'OFICIALIZADO',
 'situacao': 'EXTINTO',
 'tipo_responsavel': None,
 'data_criacao': '1997-01-23',
 'data_ultima_decisao': '2011-12-01',
 'uf': 'AC',
 'municipio': 'Acrelândia',
 'codigo_ibge': '1200013',
 'bairro': 'Centro',
 'cep': '69945000',
 'endereco': 'Rus 7 Quedas, s/n',
 'numero': None,
 'telefone': '6832351194',
 'email': 'setan1ac@tj.ac.gov.br',
 'email_mascarado': False,
 'website': None,
 'horario_funcionamento': 'domingo: Fechado; segunda-feira: 08:00 - 18:00; terça-feira: 08:00 - 18:00; quarta-feira: 08:00 - 18:00; quinta-feira: 08:00 - 18:00; sexta-feira: 08:00 - 18:00; sábado: Fechado;',
 'informatizada': False,
 'acesso_internet': False,
 'conciliacao_mediacao': False,
 'sistema_interligado': False,
 'funcionarios_clt': 1,
 'funcionarios_estatutarios': 0,
 'colaboradores_estagiarios': 0,
 'colaboradores_terceirizados': 0,
 'colaboradores_total': 1,

## Join with the municipalities

`codigo_ibge` joins [municipios-e-estados-do-brasil](https://www.kaggle.com/datasets/caiquedourado/municipios-e-estados-do-brasil)
(population, region, coordinates). On Kaggle, add that dataset to the notebook.

In [11]:
mun = glob.glob('/kaggle/input/**/municipios.parquet', recursive=True)
if mun:
    display(q(f'''SELECT m.region, count(*) serventias, round(sum(m.population_2022) / count(DISTINCT m.ibge_code)) pop_media
                  FROM {SERV} s JOIN read_parquet('{mun[0]}') m ON m.ibge_code = s.codigo_ibge GROUP BY 1 ORDER BY 2 DESC'''))
else:
    print('add municipios-e-estados-do-brasil to join by codigo_ibge')

add municipios-e-estados-do-brasil to join by codigo_ibge
